# Classification Performance Metrics

## Confusion Matrix, Accuracy, Precision, Recall and F-beta

A classification model needs more than one score. This lesson uses a small binary example to explain what each metric measures and when to prefer it.

![Confusion matrix metrics](https://miro.medium.com/v2/resize%3Afit%3A1400/1%2AAHlJoMICNZvd2YbSmI7pdg.png)

Image source: [Confusion Matrix - Clearly Explained](https://medium.com/data-science/confusion-matrix-clearly-explained-fee63614dc7). Formula reference: [scikit-learn F-beta documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.fbeta_score.html).

## 1. Start with the confusion matrix

For binary classification, label 1 is called positive and label 0 is called negative.

| Actual | Predicted | Name | Meaning |
|---|---|---|---|
| 1 | 1 | True Positive (TP) | Correctly found a positive case |
| 0 | 0 | True Negative (TN) | Correctly found a negative case |
| 0 | 1 | False Positive (FP) | False alarm |
| 1 | 0 | False Negative (FN) | Missed positive case |

In [ ]:
# Cell 1: create true labels and model predictions
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, fbeta_score

y_true = np.array([0, 1, 0, 1, 1, 0, 1])
y_pred = np.array([1, 1, 0, 1, 1, 1, 0])
pd.DataFrame({'Actual value': y_true, 'Model prediction': y_pred})

### What changed here?

y_true contains the correct answers. y_pred contains the model answers. Never swap them when calling metrics: the order is always true values first, predictions second.

In [ ]:
# Cell 2: draw the confusion matrix
matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
display = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=['Negative (0)', 'Positive (1)'])
display.plot(cmap='Blues', values_format='d')
plt.title('Rows are actual values; columns are model predictions')
plt.show()

tn, fp, fn, tp = matrix.ravel()
print(f'TN={tn}, FP={fp}, FN={fn}, TP={tp}')

## 2. Accuracy

    Accuracy = (TP + TN) / (TP + TN + FP + FN)

Accuracy answers: **Out of all predictions, how many were correct?** It is easy to understand, but it can be misleading when one class is much more common than the other.

In [ ]:
# Cell 3: calculate accuracy manually and with scikit-learn
manual_accuracy = (tp + tn) / (tp + tn + fp + fn)
print('Manual accuracy:', round(manual_accuracy, 3))
print('scikit-learn accuracy:', round(accuracy_score(y_true, y_pred), 3))

### Accuracy can hide a weak model

Imagine 900 negative examples and 100 positive examples. A lazy model that predicts negative every time gets 90% accuracy, but finds zero positive cases. Use precision, recall, and F-scores when the classes are imbalanced.

In [ ]:
# Cell 4: see why imbalanced data needs more than accuracy
imbalanced_true = np.array([0] * 900 + [1] * 100)
always_negative = np.zeros_like(imbalanced_true)
print('Accuracy of always-negative model:', accuracy_score(imbalanced_true, always_negative))
print('Recall for positive class:', recall_score(imbalanced_true, always_negative, zero_division=0))

## 3. Precision and recall

    Precision = TP / (TP + FP)
    Recall    = TP / (TP + FN)

- **Precision:** Of all predicted positives, how many were actually positive? Use it when false alarms are costly. Example: a legitimate email incorrectly sent to spam.
- **Recall:** Of all actual positives, how many did we find? Use it when missing a positive is costly. Example: a disease screening system missing a person who needs follow-up.

In [ ]:
# Cell 5: calculate precision and recall
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
print('Precision:', round(precision, 3), '= TP / (TP + FP)')
print('Recall:', round(recall, 3), '= TP / (TP + FN)')

In [ ]:
# Cell 6: visual guide for choosing a metric
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')
boxes = [
    (0.05, 'False positive is costly', 'Focus on precision', '#FADBD8'),
    (0.38, 'False negative is costly', 'Focus on recall', '#D5F5E3'),
    (0.71, 'Both matter', 'Use F1 or F-beta', '#D6EAF8')
]
for x, problem, choice, color in boxes:
    ax.text(x, 0.62, problem, transform=ax.transAxes, ha='center', va='center', fontsize=12, weight='bold', bbox={'boxstyle': 'round,pad=0.7', 'facecolor': color, 'edgecolor': '#444'})
    ax.text(x, 0.28, choice, transform=ax.transAxes, ha='center', va='center', fontsize=12, color='#1F4E79')
plt.title('Choose the metric based on the cost of mistakes', fontsize=14, weight='bold')
plt.show()

## 4. F1 and F-beta scores

F1 is the balanced harmonic mean of precision and recall:

    F1 = 2 × precision × recall / (precision + recall)

F-beta lets you give more importance to recall or precision:

    F-beta = (1 + beta squared) × precision × recall / (beta squared × precision + recall)

- beta = 1: precision and recall have equal importance.
- beta less than 1: precision has more importance.
- beta greater than 1: recall has more importance.

In [ ]:
# Cell 7: compare F1, F0.5 and F2 scores
scores = pd.DataFrame({
    'Metric': ['F0.5 (precision focus)', 'F1 (balanced)', 'F2 (recall focus)'],
    'Score': [fbeta_score(y_true, y_pred, beta=0.5), f1_score(y_true, y_pred), fbeta_score(y_true, y_pred, beta=2)]
})
scores['Score'] = scores['Score'].round(3)
scores

## Multi-class note

For three classes, the confusion matrix becomes 3 by 3; for four classes, 4 by 4. Precision and recall are first calculated per class, then can be averaged. Common averages are macro (every class equal), weighted (larger classes count more), and micro (count all decisions together).

## Quick revision card

1. Confusion matrix gives TN, FP, FN, and TP.
2. Accuracy is correct predictions divided by all predictions.
3. Avoid trusting accuracy alone on imbalanced data.
4. Precision reduces concern about false positives.
5. Recall reduces concern about false negatives.
6. F1 balances precision and recall; F-beta lets you favour one.
7. Choose a metric based on the real cost of each error.

**One-line interview answer:** Use precision when false positives are more costly, recall when false negatives are more costly, and F-beta when both matter but need different weights.